In [1]:
import sys
import os
import json
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from model import get_model
from utils import compute_metrics
from transformers import Trainer, TrainingArguments # Substitua pelo seu WeightedTrainer se precisar
from datasets import load_from_disk

/home/giulia-mezaroba/Faculdade-Codigos/Cyber-Threat-Dataset-with-Transformers/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Carregar os dados processados e dividir
dataset_full = load_from_disk('../data/processed/dataset_hf_processado')
datasets_split = dataset_full.train_test_split(test_size=0.2, seed=42)

dataset_train_hf = datasets_split['train']
dataset_test_hf = datasets_split['test']


In [3]:
# 2. Carregar o mapa de labels para injetar no modelo
with open('../data/processed/label_map.json', 'r') as f:
    id2label = json.load(f)
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)

In [4]:
# 3. Carregar o Modelo
model = get_model(num_labels=num_labels, id2label=id2label, label2id=label2id)

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 1031.28it/s]
ModernBertForSequenceClassification LOAD REPORT from: cisco-ai/SecureBERT2.0-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

# 4. Argumentos de Treino
training_args = TrainingArguments(
    output_dir="../models/securebert-cyber-threat",

    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)


In [ ]:
from transformers import TrainerCallback

# Nome do experimento (MUDE AQUI A CADA TREINO)
NOME_EXPERIMENTO = "securebert_lr2e5_ep10"

# Caminho para salvar histórico
PASTA_HIST = "../data/processed/historic"
os.makedirs(PASTA_HIST, exist_ok=True)

class SaveHistoryCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.log_history:
            df = pd.DataFrame(state.log_history)
            caminho = os.path.join(PASTA_HIST, f"{NOME_EXPERIMENTO}.csv")
            df.to_csv(caminho, index=False)

In [ ]:
import sys
import os

# 1. Ensina ao Python onde encontrar a pasta 'src'
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

# 2. Agora os imports vão funcionar perfeitamente
from transformers import Trainer
from model import get_model

# IMPORTANTE: Você também precisa importar a sua função de métricas e os dados se a memória foi limpa
from utils import compute_metrics
from datasets import load_from_disk
import json

# Recarregando os dados e variáveis essenciais rapidamente
dataset_full = load_from_disk('../data/processed/dataset_hf_processado')
datasets_split = dataset_full.train_test_split(test_size=0.2, seed=42)
dataset_train_hf = datasets_split['train']
dataset_test_hf = datasets_split['test']

with open('../data/processed/label_map.json', 'r') as f:
    id2label = json.load(f)
label2id = {v: k for k, v in id2label.items()}

# Carregando o modelo
model = get_model(num_labels=len(id2label), id2label=id2label, label2id=label2id)

# 3. Inicializar o Trainer
trainer = Trainer(
    model=model,
    args=training_args, # Assume que você já rodou a célula do training_args
    train_dataset=dataset_train_hf,
    eval_dataset=dataset_test_hf,
    compute_metrics=compute_metrics,
    callbacks=[SaveHistoryCallback()] 
)

print("Trainer inicializado com sucesso e módulos importados!")


Loading weights: 100%|██████████| 136/136 [00:00<00:00, 1382.54it/s]
ModernBertForSequenceClassification LOAD REPORT from: cisco-ai/SecureBERT2.0-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer inicializado com sucesso e módulos importados!


In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro
1,No log,0.293449,0.194541,0.383831,0.160177
2,No log,0.242972,0.409313,0.566340,0.362487
3,No log,0.212368,0.440714,0.622914,0.379023
4,No log,0.199654,0.480051,0.645202,0.420347
5,No log,0.188829,0.591482,0.727153,0.518796
6,No log,0.181967,0.597562,0.736667,0.532962
7,No log,0.186489,0.582216,0.715460,0.513749
8,No log,0.188320,0.588963,0.735509,0.518934
9,No log,0.191176,0.578915,0.712144,0.511762
10,No log,0.190851,0.576048,0.711290,0.508058


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.21s/it]


TrainOutput(global_step=240, training_loss=0.11873327891031901, metrics={'train_runtime': 234.3061, 'train_samples_per_second': 16.133, 'train_steps_per_second': 1.024, 'total_flos': 322045186176000.0, 'train_loss': 0.11873327891031901, 'epoch': 10.0})